In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
import torch

# Paths
train_path = "../Sentiment/opinion_detector_data/train.csv"
save_dir = "../Sentiment/opinion_detector_model/final"
os.makedirs(save_dir, exist_ok=True)

print("✅ Training file:", train_path)



✅ Training file: ../Sentiment/opinion_detector_data/train.csv


In [2]:
df = pd.read_csv(train_path)
print("✅ Dataset loaded:", df.shape)
print(df["label"].value_counts())
df.head()


✅ Dataset loaded: (56589, 2)
label
1    37994
0    18595
Name: count, dtype: int64


,text,label
0,I saw only the first part of this series when ...,1
1,REV,0
2,REV,0
3,The movie was very moving It was tender and fu...,1
4,it was clean room and staff was good,1


In [3]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print(f"Train size: {len(train_texts)} | Validation size: {len(val_texts)}")


Train size: 45271 | Validation size: 11318


In [4]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels})
val_ds = Dataset.from_dict({"text": val_texts, "label": val_labels})

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


Map:   0%|          | 0/45271 [00:00<?, ? examples/s]

Map:   0%|          | 0/11318 [00:00<?, ? examples/s]

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    prec = precision_score(labels, preds)
    rec = recall_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": prec, "recall": rec}


In [7]:
training_args = TrainingArguments(
    output_dir="../Sentiment/opinion_detector_model",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="../Sentiment/opinion_detector_model/logs",
    logging_steps=100,
)


C:\Users\keshu\anaconda3\Lib\site-packages\transformers\training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()


C:\Users\keshu\AppData\Local\Temp\ipykernel_10496\492363080.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.020900,0.008579,0.997614,0.998226,0.996981,0.999474
2,0.005500,0.009019,0.998410,0.998816,0.998422,0.999210


TrainOutput(global_step=5660, training_loss=0.010061989015119658, metrics={'train_runtime': 1095.5104, 'train_samples_per_second': 82.648, 'train_steps_per_second': 5.167, 'total_flos': 2998465802277888.0, 'train_loss': 0.010061989015119658, 'epoch': 2.0})

In [9]:
results = trainer.evaluate()
print("\n📊 Evaluation Results:")
for k, v in results.items():
    print(f"{k}: {v:.4f}")



📊 Evaluation Results:
eval_loss: 0.0090
eval_accuracy: 0.9984
eval_f1: 0.9988
eval_precision: 0.9984
eval_recall: 0.9992
eval_runtime: 32.9677
eval_samples_per_second: 343.3060
eval_steps_per_second: 10.7380
epoch: 2.0000


In [10]:
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"\n✅ Model saved at: {save_dir}")



✅ Model saved at: ../Sentiment/opinion_detector_model/final


In [11]:
from transformers import pipeline

classifier = pipeline("text-classification", model=save_dir, tokenizer=save_dir)

samples = [
    "The restaurant was amazing, great staff and delicious food!",
    "Order ID: 34821",
    "www.companywebsite.com",
    "Terrible product, broke after one day.",
    "New York City"
]

for s in samples:
    result = classifier(s)[0]
    print(f"{s[:50]}... → {result}")


Device set to use cuda:0


The restaurant was amazing, great staff and delici... → {'label': 'LABEL_1', 'score': 0.9999762773513794}
Order ID: 34821... → {'label': 'LABEL_0', 'score': 0.9899193048477173}
www.companywebsite.com... → {'label': 'LABEL_0', 'score': 0.9969823956489563}
Terrible product, broke after one day.... → {'label': 'LABEL_1', 'score': 0.9999822378158569}
New York City... → {'label': 'LABEL_0', 'score': 0.9999710321426392}
